In [ ]:
import pandas as pd

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# load HR data
hr_baseline = pd.read_csv(f'{DATA}/hr.csv')
hr_01 = pd.read_csv(f'{DATA}/hr_01.csv')
hr_02 = pd.read_csv(f'{DATA}/hr_02.csv')
hr_03 = pd.read_csv(f'{DATA}/hr_03.csv')

# load psychometric data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# datetime conversion
for df in [hr_baseline, hr_01, hr_02, hr_03]:
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
for df in [psychometric_01, psychometric_02, psychometric_03]:
    df['Question Start Time'] = pd.to_datetime(df['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    df['Question Answer Time'] = pd.to_datetime(df['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

# drop NaT rows
hr_baseline = hr_baseline.dropna(subset=['datetime'])
hr_01 = hr_01.dropna(subset=['datetime'])
hr_02 = hr_02.dropna(subset=['datetime'])
hr_03 = hr_03.dropna(subset=['datetime'])
psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

# helper functions
def get_average_hr_and_times(questions, hr_data):
    questions = questions.copy()
    questions.loc[:, 'closest_hr'] = questions['Question Start Time'].apply(
        lambda x: hr_data.iloc[(hr_data['datetime'] - x).abs().argsort()[:1]]['heart_rate'].values[0])
    average_hr = questions['closest_hr'].mean()
    start_time = questions['Question Start Time'].min().strftime('%H:%M:%S')
    end_time = questions['Question Answer Time'].max().strftime('%H:%M:%S')
    return average_hr, start_time, end_time

def get_average_hr_and_start_time(questions, hr_data):
    questions = questions.copy()
    questions.loc[:, 'closest_hr'] = questions['Question Start Time'].apply(
        lambda x: hr_data.iloc[(hr_data['datetime'] - x).abs().argsort()[:1]]['heart_rate'].values[0])
    average_hr = questions['closest_hr'].mean()
    start_time = questions['Question Start Time'].min()
    return average_hr, start_time

# baseline average
baseline_avg_hr = hr_baseline['heart_rate'].mean()

print('Setup complete.')
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM')

In [ ]:
# Filter '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Calculate average heart rate during 14 HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_14_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_14_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_14_hads_03, hr_03)

# Check HR
anxiety_01 = average_hr_01 > baseline_avg_hr
anxiety_02 = average_hr_02 > baseline_avg_hr
anxiety_03 = average_hr_03 > baseline_avg_hr

# Display the results
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM')
print(f'Start time for 14 HADS in Test 01: {start_time_01} - End time: {end_time_01}')
print(f'Average HR during 14 HADS in Test 01: {average_hr_01:.2f} BPM - Anxiety: {"Yes" if anxiety_01 else "No"}')
print(f'Start time for 14 HADS in Test 02: {start_time_02} - End time: {end_time_02}')
print(f'Average HR during 14 HADS in Test 02: {average_hr_02:.2f} BPM - Anxiety: {"Yes" if anxiety_02 else "No"}')
print(f'Start time for 14 HADS in Test 03: {start_time_03} - End time: {end_time_03}')
print(f'Average HR during 14 HADS in Test 03: {average_hr_03:.2f} BPM - Anxiety: {"Yes" if anxiety_03 else "No"}')

In [ ]:
# Filter for '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Average HR
average_hr_01, start_time_01 = get_average_hr_and_start_time(questions_14_hads_01, hr_01)
average_hr_02, start_time_02 = get_average_hr_and_start_time(questions_14_hads_02, hr_02)
average_hr_03, start_time_03 = get_average_hr_and_start_time(questions_14_hads_03, hr_03)

# Indicates Anxiety
anxiety_01 = average_hr_01 > baseline_avg_hr
anxiety_02 = average_hr_02 > baseline_avg_hr
anxiety_03 = average_hr_03 > baseline_avg_hr

# Display
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM')
print(f'Start time for 14 HADS in Test 01: {start_time_01}')
print(f'Average HR during 14 HADS in Test 01: {average_hr_01:.2f} BPM - Anxiety: {"Yes" if anxiety_01 else "No"}')
print(f'Start time for 14 HADS in Test 02: {start_time_02}')
print(f'Average HR during 14 HADS in Test 02: {average_hr_02:.2f} BPM - Anxiety: {"Yes" if anxiety_02 else "No"}')
print(f'Start time for 14 HADS in Test 03: {start_time_03}')
print(f'Average HR during 14 HADS in Test 03: {average_hr_03:.2f} BPM - Anxiety: {"Yes" if anxiety_03 else "No"}')

In [ ]:
# Filter '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Calculate average heart rate during 14 HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_14_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_14_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_14_hads_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_01 = average_hr_01 > baseline_avg_hr
anxiety_general_01 = average_hr_01 > 100
anxiety_individual_02 = average_hr_02 > baseline_avg_hr
anxiety_general_02 = average_hr_02 > 100
anxiety_individual_03 = average_hr_03 > baseline_avg_hr
anxiety_general_03 = average_hr_03 > 100

# Display the results
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM')
print(f'Start time for 14 HADS in Test 01: {start_time_01} - End time: {end_time_01}')
print(f'Average HR during 14 HADS in Test 01: {average_hr_01:.2f} BPM - Anxiety (Individual): {"Yes" if anxiety_individual_01 else "No"} - Anxiety (General): {"Yes" if anxiety_general_01 else "No"}')
print(f'Start time for 14 HADS in Test 02: {start_time_02} - End time: {end_time_02}')
print(f'Average HR during 14 HADS in Test 02: {average_hr_02:.2f} BPM - Anxiety (Individual): {"Yes" if anxiety_individual_02 else "No"} - Anxiety (General): {"Yes" if anxiety_general_02 else "No"}')
print(f'Start time for 14 HADS in Test 03: {start_time_03} - End time: {end_time_03}')
print(f'Average HR during 14 HADS in Test 03: {average_hr_03:.2f} BPM - Anxiety (Individual): {"Yes" if anxiety_individual_03 else "No"} - Anxiety (General): {"Yes" if anxiety_general_03 else "No"}')

In [ ]:
# Filter '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Calculate average heart rate during 14 HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_14_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_14_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_14_hads_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_01 = average_hr_01 > baseline_avg_hr
anxiety_general_01 = average_hr_01 > 100
anxiety_individual_02 = average_hr_02 > baseline_avg_hr
anxiety_general_02 = average_hr_02 > 100
anxiety_individual_03 = average_hr_03 > baseline_avg_hr
anxiety_general_03 = average_hr_03 > 100

# Show results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

print(f"Test 01:")
print(f"  Start time for 14 HADS: {start_time_01}")
print(f"  End time for 14 HADS: {end_time_01}")
print(f"  Average HR during 14 HADS: {average_hr_01:.2f} BPM")
print(f"  Anxiety (Individual Threshold): {'Yes' if anxiety_individual_01 else 'No'}")
print(f"  Anxiety (General Threshold >100 BPM): {'Yes' if anxiety_general_01 else 'No'}\n")

print(f"Test 02:")
print(f"  Start time for 14 HADS: {start_time_02}")
print(f"  End time for 14 HADS: {end_time_02}")
print(f"  Average HR during 14 HADS: {average_hr_02:.2f} BPM")
print(f"  Anxiety (Individual Threshold): {'Yes' if anxiety_individual_02 else 'No'}")
print(f"  Anxiety (General Threshold >100 BPM): {'Yes' if anxiety_general_02 else 'No'}\n")

print(f"Test 03:")
print(f"  Start time for 14 HADS: {start_time_03}")
print(f"  End time for 14 HADS: {end_time_03}")
print(f"  Average HR during 14 HADS: {average_hr_03:.2f} BPM")
print(f"  Anxiety (Individual Threshold): {'Yes' if anxiety_individual_03 else 'No'}")
print(f"  Anxiety (General Threshold >100 BPM): {'Yes' if anxiety_general_03 else 'No'}\n")

In [ ]:
# Filter '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Calculate average heart rate during 14 HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_14_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_14_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_14_hads_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_01 = average_hr_01 > baseline_avg_hr
anxiety_general_01 = average_hr_01 > 100
anxiety_individual_02 = average_hr_02 > baseline_avg_hr
anxiety_general_02 = average_hr_02 > 100
anxiety_individual_03 = average_hr_03 > baseline_avg_hr
anxiety_general_03 = average_hr_03 > 100

# Create results table
results = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_01, start_time_02, start_time_03],
    'End Time': [end_time_01, end_time_02, end_time_03],
    'Average HR (BPM)': [average_hr_01, average_hr_02, average_hr_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No']
})

print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")
print(results)

In [ ]:
# Filter '14 HADS'
questions_14_hads_01 = psychometric_01[(psychometric_01['Type'] == 'HADS') & (psychometric_01['Test'].str.contains('14'))].copy()
questions_14_hads_02 = psychometric_02[(psychometric_02['Type'] == 'HADS') & (psychometric_02['Test'].str.contains('14'))].copy()
questions_14_hads_03 = psychometric_03[(psychometric_03['Type'] == 'HADS') & (psychometric_03['Test'].str.contains('14'))].copy()

# Calculate average heart rate during 14 HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_14_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_14_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_14_hads_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_01 = average_hr_01 > baseline_avg_hr
anxiety_general_01 = average_hr_01 > 100
anxiety_individual_02 = average_hr_02 > baseline_avg_hr
anxiety_general_02 = average_hr_02 > 100
anxiety_individual_03 = average_hr_03 > baseline_avg_hr
anxiety_general_03 = average_hr_03 > 100

# Create results table
results = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_01, start_time_02, start_time_03],
    'End Time': [end_time_01, end_time_02, end_time_03],
    'Average HR (BPM)': [average_hr_01, average_hr_02, average_hr_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for 14 HADS: {row['Start Time']}")
    print(f"  End time for 14 HADS: {row['End Time']}")
    print(f"  Average HR during 14 HADS: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for HADS questions
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

# Calculate average heart rate during HADS
average_hr_01, start_time_01, end_time_01 = get_average_hr_and_times(questions_hads_01, hr_01)
average_hr_02, start_time_02, end_time_02 = get_average_hr_and_times(questions_hads_02, hr_02)
average_hr_03, start_time_03, end_time_03 = get_average_hr_and_times(questions_hads_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_01 = average_hr_01 > baseline_avg_hr
anxiety_general_01 = average_hr_01 > 100
anxiety_individual_02 = average_hr_02 > baseline_avg_hr
anxiety_general_02 = average_hr_02 > 100
anxiety_individual_03 = average_hr_03 > baseline_avg_hr
anxiety_general_03 = average_hr_03 > 100

# Create results table
results = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_01, start_time_02, start_time_03],
    'End Time': [end_time_01, end_time_02, end_time_03],
    'Average HR (BPM)': [average_hr_01, average_hr_02, average_hr_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for HADS: {row['Start Time']}")
    print(f"  End time for HADS: {row['End Time']}")
    print(f"  Average HR during HADS: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for STAI-S questions
questions_stai_s_01 = psychometric_01[psychometric_01['Type'] == 'STAI-S'].copy()
questions_stai_s_02 = psychometric_02[psychometric_02['Type'] == 'STAI-S'].copy()
questions_stai_s_03 = psychometric_03[psychometric_03['Type'] == 'STAI-S'].copy()

# Calculate average heart rate during STAI-S
average_hr_stai_s_01, start_time_stai_s_01, end_time_stai_s_01 = get_average_hr_and_times(questions_stai_s_01, hr_01)
average_hr_stai_s_02, start_time_stai_s_02, end_time_stai_s_02 = get_average_hr_and_times(questions_stai_s_02, hr_02)
average_hr_stai_s_03, start_time_stai_s_03, end_time_stai_s_03 = get_average_hr_and_times(questions_stai_s_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_stai_s_01 = average_hr_stai_s_01 > baseline_avg_hr
anxiety_general_stai_s_01 = average_hr_stai_s_01 > 100
anxiety_individual_stai_s_02 = average_hr_stai_s_02 > baseline_avg_hr
anxiety_general_stai_s_02 = average_hr_stai_s_02 > 100
anxiety_individual_stai_s_03 = average_hr_stai_s_03 > baseline_avg_hr
anxiety_general_stai_s_03 = average_hr_stai_s_03 > 100

# Create results table
results_stai_s = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_stai_s_01, start_time_stai_s_02, start_time_stai_s_03],
    'End Time': [end_time_stai_s_01, end_time_stai_s_02, end_time_stai_s_03],
    'Average HR (BPM)': [average_hr_stai_s_01, average_hr_stai_s_02, average_hr_stai_s_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_stai_s_01 else 'No', 'Yes' if anxiety_individual_stai_s_02 else 'No', 'Yes' if anxiety_individual_stai_s_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_stai_s_01 else 'No', 'Yes' if anxiety_general_stai_s_02 else 'No', 'Yes' if anxiety_general_stai_s_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results_stai_s.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for STAI-S: {row['Start Time']}")
    print(f"  End time for STAI-S: {row['End Time']}")
    print(f"  Average HR during STAI-S: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for STAI-T questions
questions_stai_t_01 = psychometric_01[psychometric_01['Type'] == 'STAI-T'].copy()
questions_stai_t_02 = psychometric_02[psychometric_02['Type'] == 'STAI-T'].copy()
questions_stai_t_03 = psychometric_03[psychometric_03['Type'] == 'STAI-T'].copy()

# Calculate average heart rate during STAI-T
average_hr_stai_t_01, start_time_stai_t_01, end_time_stai_t_01 = get_average_hr_and_times(questions_stai_t_01, hr_01)
average_hr_stai_t_02, start_time_stai_t_02, end_time_stai_t_02 = get_average_hr_and_times(questions_stai_t_02, hr_02)
average_hr_stai_t_03, start_time_stai_t_03, end_time_stai_t_03 = get_average_hr_and_times(questions_stai_t_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_stai_t_01 = average_hr_stai_t_01 > baseline_avg_hr
anxiety_general_stai_t_01 = average_hr_stai_t_01 > 100
anxiety_individual_stai_t_02 = average_hr_stai_t_02 > baseline_avg_hr
anxiety_general_stai_t_02 = average_hr_stai_t_02 > 100
anxiety_individual_stai_t_03 = average_hr_stai_t_03 > baseline_avg_hr
anxiety_general_stai_t_03 = average_hr_stai_t_03 > 100

# Create results table
results_stai_t = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_stai_t_01, start_time_stai_t_02, start_time_stai_t_03],
    'End Time': [end_time_stai_t_01, end_time_stai_t_02, end_time_stai_t_03],
    'Average HR (BPM)': [average_hr_stai_t_01, average_hr_stai_t_02, average_hr_stai_t_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_stai_t_01 else 'No', 'Yes' if anxiety_individual_stai_t_02 else 'No', 'Yes' if anxiety_individual_stai_t_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_stai_t_01 else 'No', 'Yes' if anxiety_general_stai_t_02 else 'No', 'Yes' if anxiety_general_stai_t_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results_stai_t.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for STAI-T: {row['Start Time']}")
    print(f"  End time for STAI-T: {row['End Time']}")
    print(f"  Average HR during STAI-T: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for BFI questions
questions_bfi_01 = psychometric_01[psychometric_01['Type'] == 'BFI'].copy()
questions_bfi_02 = psychometric_02[psychometric_02['Type'] == 'BFI'].copy()
questions_bfi_03 = psychometric_03[psychometric_03['Type'] == 'BFI'].copy()

# Calculate average heart rate during BFI
average_hr_bfi_01, start_time_bfi_01, end_time_bfi_01 = get_average_hr_and_times(questions_bfi_01, hr_01)
average_hr_bfi_02, start_time_bfi_02, end_time_bfi_02 = get_average_hr_and_times(questions_bfi_02, hr_02)
average_hr_bfi_03, start_time_bfi_03, end_time_bfi_03 = get_average_hr_and_times(questions_bfi_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_bfi_01 = average_hr_bfi_01 > baseline_avg_hr
anxiety_general_bfi_01 = average_hr_bfi_01 > 100
anxiety_individual_bfi_02 = average_hr_bfi_02 > baseline_avg_hr
anxiety_general_bfi_02 = average_hr_bfi_02 > 100
anxiety_individual_bfi_03 = average_hr_bfi_03 > baseline_avg_hr
anxiety_general_bfi_03 = average_hr_bfi_03 > 100

# Create results table
results_bfi = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_bfi_01, start_time_bfi_02, start_time_bfi_03],
    'End Time': [end_time_bfi_01, end_time_bfi_02, end_time_bfi_03],
    'Average HR (BPM)': [average_hr_bfi_01, average_hr_bfi_02, average_hr_bfi_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_bfi_01 else 'No', 'Yes' if anxiety_individual_bfi_02 else 'No', 'Yes' if anxiety_individual_bfi_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_bfi_01 else 'No', 'Yes' if anxiety_general_bfi_02 else 'No', 'Yes' if anxiety_general_bfi_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results_bfi.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for BFI: {row['Start Time']}")
    print(f"  End time for BFI: {row['End Time']}")
    print(f"  Average HR during BFI: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for BFI questions
questions_bfi_01 = psychometric_01[psychometric_01['Type'] == 'BFI'].copy()
questions_bfi_02 = psychometric_02[psychometric_02['Type'] == 'BFI'].copy()
questions_bfi_03 = psychometric_03[psychometric_03['Type'] == 'BFI'].copy()

# Calculate average heart rate during BFI
average_hr_bfi_01, start_time_bfi_01, end_time_bfi_01 = get_average_hr_and_times(questions_bfi_01, hr_01)
average_hr_bfi_02, start_time_bfi_02, end_time_bfi_02 = get_average_hr_and_times(questions_bfi_02, hr_02)
average_hr_bfi_03, start_time_bfi_03, end_time_bfi_03 = get_average_hr_and_times(questions_bfi_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_bfi_01 = average_hr_bfi_01 > baseline_avg_hr
anxiety_general_bfi_01 = average_hr_bfi_01 > 100
anxiety_individual_bfi_02 = average_hr_bfi_02 > baseline_avg_hr
anxiety_general_bfi_02 = average_hr_bfi_02 > 100
anxiety_individual_bfi_03 = average_hr_bfi_03 > baseline_avg_hr
anxiety_general_bfi_03 = average_hr_bfi_03 > 100

# Create results table
results_bfi = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_bfi_01, start_time_bfi_02, start_time_bfi_03],
    'End Time': [end_time_bfi_01, end_time_bfi_02, end_time_bfi_03],
    'Average HR (BPM)': [average_hr_bfi_01, average_hr_bfi_02, average_hr_bfi_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_bfi_01 else 'No', 'Yes' if anxiety_individual_bfi_02 else 'No', 'Yes' if anxiety_individual_bfi_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_bfi_01 else 'No', 'Yes' if anxiety_general_bfi_02 else 'No', 'Yes' if anxiety_general_bfi_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results_bfi.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for BFI: {row['Start Time']}")
    print(f"  End time for BFI: {row['End Time']}")
    print(f"  Average HR during BFI: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Filter for FQ questions
questions_fq_01 = psychometric_01[psychometric_01['Type'] == 'FQ'].copy()
questions_fq_02 = psychometric_02[psychometric_02['Type'] == 'FQ'].copy()
questions_fq_03 = psychometric_03[psychometric_03['Type'] == 'FQ'].copy()

# Calculate average heart rate during FQ
average_hr_fq_01, start_time_fq_01, end_time_fq_01 = get_average_hr_and_times(questions_fq_01, hr_01)
average_hr_fq_02, start_time_fq_02, end_time_fq_02 = get_average_hr_and_times(questions_fq_02, hr_02)
average_hr_fq_03, start_time_fq_03, end_time_fq_03 = get_average_hr_and_times(questions_fq_03, hr_03)

# Check HR anxiety threshold
anxiety_individual_fq_01 = average_hr_fq_01 > baseline_avg_hr
anxiety_general_fq_01 = average_hr_fq_01 > 100
anxiety_individual_fq_02 = average_hr_fq_02 > baseline_avg_hr
anxiety_general_fq_02 = average_hr_fq_02 > 100
anxiety_individual_fq_03 = average_hr_fq_03 > baseline_avg_hr
anxiety_general_fq_03 = average_hr_fq_03 > 100

# Create results table
results_fq = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [start_time_fq_01, start_time_fq_02, start_time_fq_03],
    'End Time': [end_time_fq_01, end_time_fq_02, end_time_fq_03],
    'Average HR (BPM)': [average_hr_fq_01, average_hr_fq_02, average_hr_fq_03],
    'Anxiety (Individual)': ['Yes' if anxiety_individual_fq_01 else 'No', 'Yes' if anxiety_individual_fq_02 else 'No', 'Yes' if anxiety_individual_fq_03 else 'No'],
    'Anxiety (General >100 BPM)': ['Yes' if anxiety_general_fq_01 else 'No', 'Yes' if anxiety_general_fq_02 else 'No', 'Yes' if anxiety_general_fq_03 else 'No']
})

# Display results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")

for index, row in results_fq.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for FQ: {row['Start Time']}")
    print(f"  End time for FQ: {row['End Time']}")
    print(f"  Average HR during FQ: {row['Average HR (BPM)']:.2f} BPM")
    print(f"  Anxiety (Individual Threshold): {row['Anxiety (Individual)']}")
    print(f"  Anxiety (General Threshold >100 BPM): {row['Anxiety (General >100 BPM)']}\n")

In [ ]:
# Calculate and display results for all test types
test_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']
test_data = []
for test_type in test_types:
    for i in range(1, 4):
        psychometric = globals()[f'psychometric_0{i}']
        hr_data = globals()[f'hr_0{i}']
        questions = psychometric[psychometric['Type'] == test_type].copy()
        avg_hr, start_time, end_time = get_average_hr_and_times(questions, hr_data)
        individual_anxiety = 'Yes' if avg_hr > baseline_avg_hr else 'No'
        general_anxiety = 'Yes' if avg_hr > 100 else 'No'
        test_data.append((f'Test {i}', test_type, start_time, end_time, avg_hr, individual_anxiety, general_anxiety))

# Create results table
results = pd.DataFrame(test_data, columns=['Test', 'Type', 'Start Time', 'End Time', 'Average HR (BPM)', 'Anxiety (Individual)', 'Anxiety (General >100 BPM)'])

# Display the results
print(f"Baseline Average HR: {baseline_avg_hr:.2f} BPM\n")
print(results.to_string(index=False))